# Notebook 03 — Strategic Scoring Model

**Project:** EU Funding Intelligence — EIF Financial Intermediaries  
**Purpose:** Score and rank each fund manager based on fit with company projects

---

## Scoring Methodology

Each fund manager is scored out of **10 points** across 5 dimensions:

| Dimension | Max Points | Logic |
|---|---|---|
| Project fit | 3 pts | 1 pt per project match (Solar, Hydrogen, Desalination) |
| EIF commitment size | 2 pts | Larger commitment = more capacity |
| SDG alignment | 2 pts | More SDGs covered = broader impact fit |
| Contact completeness | 2 pts | Email + phone + website available |
| Geographic fit | 1 pt | Spain/Portugal/Global scores higher for our use case |

---

This is a **transparent, adjustable scoring model** — weights can be changed based on company priorities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../01_data/processed/eif_fund_managers_clean.csv')
print(f'Dataset loaded: {df.shape[0]} fund managers')

## 1. Score: Project Fit (max 3 points)

In [ ]:
df['score_project_fit'] = df['solar_fit'] + df['hydrogen_fit'] + df['desalination_fit']
print('Project fit scores:')
print(df[['manager_name', 'solar_fit', 'hydrogen_fit', 'desalination_fit', 'score_project_fit']].to_string())

## 2. Score: EIF Commitment Size (max 2 points)

In [ ]:
def score_commitment(amount):
    if pd.isna(amount):
        return 0
    if amount >= 40_000_000:
        return 2  # Large commitment
    elif amount >= 20_000_000:
        return 1  # Medium commitment
    return 0.5    # Small commitment

df['score_commitment'] = df['eif_commitment_eur'].apply(score_commitment)
print('Commitment scores:')
print(df[['manager_name', 'eif_commitment_eur', 'score_commitment']].to_string())

## 3. Score: SDG Alignment (max 2 points)

In [ ]:
def score_sdg(sdg_count):
    if sdg_count >= 3:
        return 2
    elif sdg_count >= 1:
        return 1
    return 0

df['score_sdg'] = df['sdg_count'].apply(score_sdg)
print('SDG alignment scores:')
print(df[['manager_name', 'sdg_count', 'score_sdg']].to_string())

## 4. Score: Contact Completeness (max 2 points)

In [ ]:
def score_contact(row):
    score = 0
    if pd.notna(row['email']) and row['email']:
        score += 0.75
    if pd.notna(row['phone']) and row['phone']:
        score += 0.75
    if pd.notna(row['website']) and row['website']:
        score += 0.5
    return min(score, 2)  # Cap at 2

df['score_contact'] = df.apply(score_contact, axis=1)
print('Contact completeness scores:')
print(df[['manager_name', 'email', 'phone', 'website', 'score_contact']].to_string())

## 5. Score: Geographic Fit (max 1 point)

In [ ]:
priority_countries = ['Spain', 'Spain & Portugal', 'Portugal', 'Global', 'Worldwide & Spain']

df['score_geography'] = df['country_primary'].apply(
    lambda c: 1 if c in priority_countries else 0.5
)

print('Geography scores:')
print(df[['manager_name', 'country_primary', 'score_geography']].to_string())

## 6. Final Composite Score

In [ ]:
df['fit_score'] = (
    df['score_project_fit'] +
    df['score_commitment'] +
    df['score_sdg'] +
    df['score_contact'] +
    df['score_geography']
).round(1)

# Normalize to 0-10 scale (max possible = 10)
df['fit_score_10'] = (df['fit_score'] / 10 * 10).round(1)

df_ranked = df.sort_values('fit_score_10', ascending=False)

print('=== FUND MANAGER RANKINGS ===')
print(df_ranked[['manager_name', 'country_primary', 'fund_type',
                  'score_project_fit', 'score_commitment', 'score_sdg',
                  'score_contact', 'score_geography', 'fit_score_10']].to_string())

## 7. Visualize Rankings

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

score_cols = ['score_project_fit', 'score_commitment', 'score_sdg', 'score_contact', 'score_geography']
score_labels = ['Project Fit', 'EIF Commitment', 'SDG Alignment', 'Contact Info', 'Geography']
colors = ['#003399', '#0066CC', '#3399FF', '#66B2FF', '#99CCFF']

bottoms = np.zeros(len(df_ranked))
for col, label, color in zip(score_cols, score_labels, colors):
    ax.barh(df_ranked['manager_name'], df_ranked[col], left=bottoms, label=label, color=color)
    bottoms += df_ranked[col].values

# Add total score labels
for i, (_, row) in enumerate(df_ranked.iterrows()):
    ax.text(row['fit_score_10'] + 0.1, i, f"{row['fit_score_10']}/10", va='center', fontsize=9)

ax.set_title('Fund Manager Strategic Fit Score (out of 10)', fontsize=13, fontweight='bold')
ax.set_xlabel('Score')
ax.set_xlim(0, 12)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../03_powerbi/screenshots/strategic_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Priority Tiers

In [ ]:
def assign_tier(score):
    if score >= 7:
        return 'Tier 1 — Priority'
    elif score >= 4:
        return 'Tier 2 — Secondary'
    return 'Tier 3 — Monitor'

df_ranked['priority_tier'] = df_ranked['fit_score_10'].apply(assign_tier)

print('=== PRIORITY TIERS ===')
for tier in ['Tier 1 — Priority', 'Tier 2 — Secondary', 'Tier 3 — Monitor']:
    tier_funds = df_ranked[df_ranked['priority_tier'] == tier]['manager_name'].tolist()
    print(f'\n{tier} ({len(tier_funds)} funds):')
    for f in tier_funds:
        score = df_ranked.loc[df_ranked['manager_name'] == f, 'fit_score_10'].values[0]
        print(f'  • {f} — Score: {score}/10')

## 9. Export Scored Dataset

In [ ]:
output_path = '../01_data/processed/eif_fund_managers_clean.csv'
df_ranked.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Scored dataset saved to: {output_path}')
print('\nFinal columns added: fit_score_10, priority_tier, score_*')
print('\nThis file is ready to import into Power BI.')

---

## Summary

The scoring model ranks all 12 fund managers across 5 strategic dimensions. The output CSV now includes:
- `fit_score_10` — Overall score out of 10
- `priority_tier` — Tier 1 / 2 / 3 classification
- Individual dimension scores for transparency

**Next step:** Import the clean CSV into Power BI to build the interactive dashboard. See `03_powerbi/README.md` for instructions.